# Handling Duplicates

**Topic:** Exploratory Data Analysis

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import Output, VBox, HBox, Button, HTML

from IPython.display import display, clear_output
from scipy import stats

titanic = sns.load_dataset("titanic")
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this session you will be able to:

- **Detect** exact duplicate rows in a DataFrame using pandas
- **Distinguish** between exact duplicates and near-duplicates that require judgment
- **Apply** the correct deduplication strategy and verify the result

> **Tip:** Use the buttons in order: Detect, Inspect, then Drop. Each step reveals something new about how duplicates are identified and removed. Hit Reset to start over.

---
## How we got here

In `04_missing_data.ipynb` we handled what was absent from the data. Now we look for something different: rows that appear more than once. Duplicate records inflate counts, distort statistics, and give models false confidence by letting them see the same example twice during training.

---
## Why this matters for data science

A single duplicate row is usually harmless. But in large datasets, duplicates can accumulate from data pipeline errors, multiple data sources being merged, or form submissions being recorded twice. When a model trains on duplicated rows, it effectively memorizes those examples, which can inflate accuracy metrics while hurting generalization to new data.

---
## Try it yourself

In [ ]:
# Build a sample DataFrame with introduced duplicates
ORIGINAL_SAMPLE = pd.DataFrame({
    "name":     ["Allen, Mr. William Henry", "Braund, Mr. Owen Harris",
                 "Cumings, Mrs. John Bradley", "Braund, Mr. Owen Harris",
                 "Heikkinen, Miss. Laina", "Allen, Mr. William Henry",
                 "McCarthy, Mr. Timothy J", "Palsson, Master. Gosta Leonard"],
    "age":      [35, 22, 38, 22, 26, 35, 54, 2],
    "pclass":   [3, 3, 1, 3, 3, 3, 1, 3],
    "ticket":   ["373450", "A/5 21171", "PC 17599", "A/5 21171",
                 "STON/O2. 3101282", "373450", "17463", "349909"],
    "survived": [0, 0, 1, 0, 1, 0, 0, 1],
})

state = {"df": ORIGINAL_SAMPLE.copy(), "step": "original"}

out = Output()

btn_detect = Button(description="Detect duplicates", button_style="info",
                    layout=widgets.Layout(width="180px"))
btn_inspect = Button(description="Inspect duplicates", button_style="warning",
                     layout=widgets.Layout(width="180px"))
btn_drop = Button(description="Drop duplicates", button_style="danger",
                  layout=widgets.Layout(width="180px"))
btn_reset = Button(description="Reset", button_style="",
                   layout=widgets.Layout(width="120px"))

def render_state():
    df = state["df"]
    step = state["step"]
    n_dupes = df.duplicated().sum()

    if step == "original":
        caption = f"Original sample — {len(df)} rows"
        color = "#212529"
    elif step == "detected":
        caption = f"Duplicate check: {n_dupes} exact duplicate row(s) found (shown in orange below)"
        color = PALETTE["secondary"]
    elif step == "inspected":
        caption = f"Showing only duplicate rows ({n_dupes} row(s))"
        color = PALETTE["secondary"]
    elif step == "dropped":
        caption = f"After drop_duplicates(): {len(df)} rows remain"
        color = PALETTE["accent"]
    else:
        caption = ""
        color = "#212529"

    with out:
        clear_output(wait=True)
        display(HTML(
            f'<div style="font-family: Inter, Arial, sans-serif; font-size: 14px; '
            f'color: {color}; margin-bottom: 8px;"><strong>{caption}</strong></div>'
        ))
        display(df)

def on_detect(b):
    state["df"] = ORIGINAL_SAMPLE.copy()
    state["step"] = "detected"
    render_state()

def on_inspect(b):
    state["df"] = ORIGINAL_SAMPLE[ORIGINAL_SAMPLE.duplicated(keep=False)].copy()
    state["step"] = "inspected"
    render_state()

def on_drop(b):
    state["df"] = ORIGINAL_SAMPLE.drop_duplicates().reset_index(drop=True)
    state["step"] = "dropped"
    render_state()

def on_reset(b):
    state["df"] = ORIGINAL_SAMPLE.copy()
    state["step"] = "original"
    render_state()

btn_detect.on_click(on_detect)
btn_inspect.on_click(on_inspect)
btn_drop.on_click(on_drop)
btn_reset.on_click(on_reset)

display(VBox([HBox([btn_detect, btn_inspect, btn_drop, btn_reset]), out]))
render_state()

---
## What's happening?

Pandas identifies exact duplicates, meaning rows where every column value matches exactly.

```python
# Count duplicates
df.duplicated().sum()

# Show only the duplicate rows (keep all copies)
df[df.duplicated(keep=False)]

# Drop duplicates, keeping the first occurrence
df.drop_duplicates()

# Drop duplicates based on specific columns only
df.drop_duplicates(subset=["name", "age", "ticket"])
```

The `keep` parameter in `duplicated()` controls which copy is flagged:
- `keep='first'` (default): marks all duplicates except the first occurrence
- `keep='last'`: marks all duplicates except the last occurrence
- `keep=False`: marks every copy of any duplicated row

Near-duplicates (same passenger, slightly different spelling of name) require fuzzy matching tools like `fuzzywuzzy` or record linkage, which are beyond the scope of basic EDA.

---
## Real-world example: Titanic Dataset

The Titanic dataset is already clean enough that it has no exact duplicates. The chart below confirms this. The before and after row counts are identical.

Notice:
- The Titanic dataset has **0 exact duplicate rows** out of 891
- This is unusually clean for a real-world dataset. In practice, duplicates are common
- Even without exact duplicates, we should check for near-duplicates based on name, age, and ticket number

> **Discussion question:** If two passengers had the same name, age, and ticket number, would that definitely be a duplicate record? What else would you check to be sure?

In [ ]:
# Before / after deduplication on the Titanic dataset
n_before = len(titanic)
n_after = len(titanic.drop_duplicates())
n_dupes = n_before - n_after

fig = go.Figure(
    data=[go.Bar(
        x=["Before deduplication", "After deduplication"],
        y=[n_before, n_after],
        marker_color=[PALETTE["secondary"], PALETTE["accent"]],
        text=[str(n_before), str(n_after)],
        textposition="outside",
        width=0.4,
    )],
    layout=base_layout(
        title=f"Titanic Row Count Before and After Deduplication (duplicates found: {n_dupes})",
        xaxis_title="",
        yaxis_title="Number of Rows",
    ),
)
fig.update_layout(showlegend=False, yaxis=dict(range=[0, 960]))
fig.show()

### Duplicate detection patterns across real-world datasets

| Dataset | Duplicate type | Detection approach |
|---------|---------------|-------------------|
| Customer database | Same email, different name casing | Lowercase all strings before dedup |
| Transaction log | Same order ID submitted twice | Dedup on order_id + timestamp |
| Survey responses | Same respondent, re-submitted | Dedup on respondent_id, keep latest |
| Merged data sources | Same entity from two feeds | Block on key fields, use fuzzy matching |
| Web scrape data | Same article from two scraped pages | Dedup on URL or content hash |

---
## Key takeaway

> **Exact duplicate detection takes one line of code; the harder judgment is deciding whether two similar-but-not-identical rows represent the same real-world event or two genuinely different ones.**

---
*Next up: 06_outlier_detection.ipynb — identifying extreme values and deciding whether they are errors or real signals*